# GAMMA DNA Tutorial 3 — Water Drilling Incident Classification

Replicating the FACET *Water Drilling Incident Classification* tutorial using **GAMMA_DNA**.

| | |
|---|---|
| **Dataset** | `water_drilling_classification_data.csv` (semicolon-delimited) |
| **Target** | `Incident` (1 = incident occurred, 0 = no incident) |
| **Key insight** | `Inverse_Rate_of_Penetration` is perfectly redundant with `Rate_of_Penetration` — we detect and remove it |


In [1]:
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DAPS_BRIX = "C:/Users/Dai Dennis/OneDrive - The Boston Consulting Group, Inc/Desktop/DAPS_Brix"
if DAPS_BRIX not in sys.path:
    sys.path.insert(0, DAPS_BRIX)

BASE_URL = (
    "https://raw.githubusercontent.com/BCG-X-Official/facet/"
    "aada69350b085f8a3dae7900b16db6c79b146f60/sphinx/source/tutorial/"
)

df = pd.read_csv(BASE_URL + "water_drilling_classification_data.csv", sep=";")

# Clean column names: remove special chars, replace spaces with underscores
df.columns = (
    df.columns
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

# Target to int
df["Incident"] = df["Incident"].astype(int)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget distribution:\n{df['Incident'].value_counts()}")


Shape: (500, 10)
Columns: ['Weight_on_bit_kg', 'Rotation_speed_rpm', 'Depth_of_operation_m', 'Mud_density_kgL', 'Rate_of_Penetration_fth', 'Temperature_C', 'Mud_Flow_in_m3s', 'Hole_diameter_m', 'Incident', 'Inverse_Rate_of_Penetration_hft']

Target distribution:
Incident
1    259
0    241
Name: count, dtype: int64


## Exploratory Data Analysis

All features are numeric. Note that the dataset contains both
**Rate of Penetration** and its mathematical inverse — a perfect redundancy.


In [2]:
df.describe().round(3)


,Weight_on_bit_kg,Rotation_speed_rpm,Depth_of_operation_m,Mud_density_kgL,Rate_of_Penetration_fth,Temperature_C,Mud_Flow_in_m3s,Hole_diameter_m,Incident,Inverse_Rate_of_Penetration_hft
count,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000
mean,301.567,7398.649,724.230,1.988,26.592,52.524,49.348,4.989,0.518,0.039
std,60.615,2458.624,280.798,0.563,4.825,18.200,28.400,1.767,0.500,0.008
min,100.000,900.000,0.000,0.500,10.000,0.000,0.000,0.500,0.000,0.025
25%,261.537,5775.351,525.893,1.612,23.451,41.166,25.540,3.753,0.000,0.033
50%,298.612,7362.721,731.832,1.942,26.735,52.411,48.437,4.988,1.000,0.037
75%,344.249,9108.649,907.924,2.354,29.936,65.546,73.583,6.222,1.000,0.043
max,500.000,15000.000,1500.000,4.000,40.000,100.000,100.000,10.000,1.000,0.100


In [3]:
# Identify ROP columns
rop_cols = [c for c in df.columns if "Rate" in c or "Penetration" in c]
print("ROP-related columns:", rop_cols)

if len(rop_cols) == 2:
    corr_val = df[rop_cols].corr().iloc[0, 1]
    print(f"Correlation between them: {corr_val:.6f}")
    print("(Perfect negative correlation = one is the reciprocal of the other)")


ROP-related columns: ['Rate_of_Penetration_fth', 'Inverse_Rate_of_Penetration_hft']
Correlation between them: -0.952342
(Perfect negative correlation = one is the reciprocal of the other)


## Model Training — Random Forest


In [4]:
from gamma.pipeline import GAMMA_DNA

g = GAMMA_DNA(df, target="Incident")
result = g.train(model_type="random_forest")
result.summary()



  ModelResult: random_forest  [binary_classification]
  Metric                                Train         Test
  --------------------------------------------------------
  accuracy                             1.0000       0.8100
  f1                                   1.0000       0.8099
  precision                            1.0000       0.8139
  recall                               1.0000       0.8100
  roc_auc                              1.0000       0.9093


In [5]:
result.plot()
print(f"\nTest AUC  : {result.roc_auc:.4f}")
print(f"Accuracy  : {result.accuracy:.4f}")
print(f"F1 Score  : {result.f1:.4f}")



Test AUC  : 0.9093
Accuracy  : 0.8100
F1 Score  : 0.8099


## SHAP Feature Importance


In [6]:
report = g.explain(compute_shap=True, compute_permutation=False)
report.plot_overview()


  Computing SHAP values on 9 features… (may take 10-60s)
  Done.


In [7]:
imp = report.to_frame()
imp.sort_values("score", ascending=False)


,model_imp,perm_imp_mean,shap_mean_abs,rank,score
feature,,,,,
Rate_of_Penetration_fth,0.189406,NaN,0.096715,1,0.096715
Inverse_Rate_of_Penetration_hft,0.167912,NaN,0.089774,2,0.089774
Depth_of_operation_m,0.092628,NaN,0.046462,3,0.046462
Mud_density_kgL,0.093331,NaN,0.043236,4,0.043236
Hole_diameter_m,0.089233,NaN,0.042778,5,0.042778
Weight_on_bit_kg,0.117726,NaN,0.031285,6,0.031285
Rotation_speed_rpm,0.109986,NaN,0.029666,7,0.029666
Mud_Flow_in_m3s,0.072715,NaN,0.027126,8,0.027126
Temperature_C,0.067062,NaN,0.016110,9,0.016110


## Feature Synergy Analysis


In [8]:
report.plot_synergy(kind="both", sample_size=200)


## Feature Redundancy Analysis

We expect the Inverse ROP column to show **maximum redundancy** with ROP,
since one is literally `1 / other`.


In [9]:
report.plot_redundancy(kind="both")


## Drop Redundant Feature & Retrain

The redundancy analysis confirms `Inverse_Rate_of_Penetration` should be dropped.
We use `g.clean()` to remove it in-place and retrain.


In [10]:
# Identify the exact inverse-ROP column name
inv_rop_col = [c for c in g.feature_cols if "Inverse" in c or "inverse" in c.lower()]
print("Dropping:", inv_rop_col)

g.clean(drop_cols=inv_rop_col)

result2 = g.train(model_type="random_forest")
print(f"\nAUC (with Inverse ROP) : {result.roc_auc:.4f}")
print(f"AUC (without)          : {result2.roc_auc:.4f}")

report2 = g.explain(compute_shap=True, compute_permutation=False)
report2.plot_redundancy(kind="both")


Dropping: ['Inverse_Rate_of_Penetration_hft']



AUC (with Inverse ROP) : 0.9093
AUC (without)          : 0.9341
  Computing SHAP values on 8 features… (may take 10-60s)


  Done.


## Partial Dependence Plots


In [11]:
# Find the ROP column name (non-inverse)
rop_col = [c for c in report2.feature_cols
           if ("Rate" in c or "Penetration" in c) and "Inverse" not in c][0]
print(f"Using column: {rop_col}")
report2.plot_pdp(rop_col, grid_points=30)


Using column: Rate_of_Penetration_fth


In [12]:
depth_col = [c for c in report2.feature_cols if "Depth" in c or "depth" in c.lower()][0]
print(f"Using column: {depth_col}")
report2.plot_pdp(depth_col, grid_points=30)


Using column: Depth_of_operation_m


## Simulation — Rate of Penetration Impact

How does incident probability change across the operating range of Rate of Penetration?


In [13]:
rop_min = g.df[rop_col].min()
rop_max = g.df[rop_col].max()
rop_grid = np.linspace(rop_min, rop_max, 25)
print(f"ROP range: [{rop_min:.2f}, {rop_max:.2f}]")
report2.plot_simulate(rop_col, rop_grid)


ROP range: [10.00, 40.00]


In [14]:
depth_min = g.df[depth_col].min()
depth_max = g.df[depth_col].max()
depth_grid = np.linspace(depth_min, depth_max, 20)
report2.plot_simulate(depth_col, depth_grid)


## Top SHAP Feature Interactions


In [15]:
report2.plot_top_interactions(top_k=10, sample_size=200)


In [16]:
top_int = report2.top_interactions(top_k=10, sample_size=200)
top_int


,feature_a,feature_b,mean_abs_interaction
0,Weight_on_bit_kg,Rotation_speed_rpm,0.029792
1,Rotation_speed_rpm,Rate_of_Penetration_fth,0.009744
2,Weight_on_bit_kg,Rate_of_Penetration_fth,0.009196
3,Rate_of_Penetration_fth,Mud_Flow_in_m3s,0.008639
4,Mud_density_kgL,Rate_of_Penetration_fth,0.008087
5,Depth_of_operation_m,Rate_of_Penetration_fth,0.007741
6,Rate_of_Penetration_fth,Hole_diameter_m,0.007314
7,Rate_of_Penetration_fth,Temperature_C,0.005968
8,Rotation_speed_rpm,Hole_diameter_m,0.005573
9,Weight_on_bit_kg,Depth_of_operation_m,0.005412


## Facet View — Redundancy & Synergy Side-by-Side


In [17]:
report2.plot_facet(sample_size=200)


## Redundancy Linkage Chart


In [18]:
report2.plot_redundancy(kind="linkage")
